# Train and evaluate small CNN classifier

This notebook trains a compact custom CNN on annotated beach-cell matrices. It uses split metadata from `Data/Preprocessed`, loads five channels per target cell with optional neighboring-cell context from `Data/Raw/beach_all`, applies the same training-only spatial augmentation as the ConvNeXt workflow, and evaluates the test set with per-class precision, recall, and F1 score.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").exists()
    or (PROJECT_ROOT / "Data").exists()
    or (PROJECT_ROOT / "pmaciag-volume-beach").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "pmaciag-volume-beach").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pmaciag-volume-beach"
if not ((PROJECT_ROOT / "Source").exists() or (PROJECT_ROOT / "Data").exists()):
    raise RuntimeError("Could not locate pmaciag-volume-beach project directory.")

source_path = PROJECT_ROOT / "Source"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.beach_dataset import (
    BeachCellDataset,
    FEATURE_SUFFIXES,
    class_weights,
    compute_channel_stats,
    find_missing_files,
    make_class_mapping,
    read_class_name_mapping,
    read_split_csv,
    summarize_nonfinite_values,
)
from Source.grid_search import run_grid_search
from Source.models import build_small_cnn
from Source.training import classification_metrics, fit_model, get_device, predict

In [2]:
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
CLASS_MAPPING_PATH = PROJECT_ROOT / "Data" / "Raw" / "Classes.xlsx"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "small_cnn"

TRAIN_SPLIT = PREPROCESSED_DIR / "train_split.csv"
VALIDATION_SPLIT = PREPROCESSED_DIR / "validation_split.csv"
TEST_SPLIT = PREPROCESSED_DIR / "test_split.csv"

IMAGE_SIZE = None
BATCH_SIZE = 32
NUM_WORKERS = 8
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4
USE_SPATIAL_AUGMENTATION = True
BASE_PARAMETER_GRID = [
    {"learning_rate": 1e-4, "weight_decay": 1e-4, "dropout": 0.25},
    {"learning_rate": 3e-5, "weight_decay": 1e-4, "dropout": 0.25},
    {"learning_rate": 1e-4, "weight_decay": 1e-3, "dropout": 0.40},
    {"learning_rate": 3e-5, "weight_decay": 1e-3, "dropout": 0.40},
]
CONTEXT_RADII = [0, 1, 2]
PARAMETER_GRID = [
    {**config, "context_radius": context_radius}
    for context_radius in CONTEXT_RADII
    for config in BASE_PARAMETER_GRID
]
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

print(f"Project root:    {PROJECT_ROOT}", flush=True)
print(f"Cell data root:  {CELL_DATA_ROOT}", flush=True)
print(f"Class mapping:   {CLASS_MAPPING_PATH}", flush=True)
print(f"Output dir:      {OUTPUT_DIR}", flush=True)
print(f"Feature order:   {FEATURE_SUFFIXES}", flush=True)

Project root:    /workspace/pmaciag-volume-beach
Cell data root:  /workspace/pmaciag-volume-beach/Data/Raw/beach_all
Class mapping:   /workspace/pmaciag-volume-beach/Data/Raw/Classes.xlsx
Output dir:      /workspace/pmaciag-volume-beach/Results/small_cnn
Feature order:   ('r', 'g', 'b', 's', 'd')


In [3]:
print("Loading dataset splits and class mapping...", flush=True)
train_df = read_split_csv(TRAIN_SPLIT)
validation_df = read_split_csv(VALIDATION_SPLIT)
test_df = read_split_csv(TEST_SPLIT)

class_to_idx = make_class_mapping(train_df, validation_df, test_df)
class_name_by_code = read_class_name_mapping(CLASS_MAPPING_PATH, class_to_idx.keys())
idx_to_class_code = {idx: label for label, idx in class_to_idx.items()}
idx_to_class_name = {
    idx: f"{code}: {class_name_by_code.get(code, f'class_{code}')}"
    for idx, code in idx_to_class_code.items()
}

class_table = pd.DataFrame(
    [
        {"encoded_class": code, "model_index": idx, "class_name": class_name_by_code.get(code, f"class_{code}")}
        for code, idx in class_to_idx.items()
    ]
).sort_values("model_index")

if all(name == f"class_{code}" for code, name in class_name_by_code.items()):
    print(
        "No class-name mapping was detected in Classes.xlsx; "
        "using fallback labels class_0, class_1, ..."
    )

print(f"Train rows:      {len(train_df):,}")
print(f"Validation rows: {len(validation_df):,}")
print(f"Test rows:       {len(test_df):,}")
display(class_table)

Loading dataset splits and class mapping...
Train rows:      3,456
Validation rows: 741
Test rows:       741


,encoded_class,model_index,class_name
0,0,0,Snow
1,1,1,Dry sediment
2,2,2,Wet sediment
3,3,3,Water
4,4,4,Bedrock
5,5,5,Boulder
6,6,6,Driftwood
7,7,7,Algae
8,8,8,Ice
9,9,9,Anthropogenic


In [4]:
print("Checking external cell data root and a small file sample...", flush=True)
all_splits_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)

if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(
        f"External cell data directory not found: {CELL_DATA_ROOT}. "
        "Mount the external drive before running training."
    )

sanity_rows = all_splits_df.head(min(20, len(all_splits_df)))
missing_files = find_missing_files(sanity_rows, CELL_DATA_ROOT)

if not missing_files.empty:
    display(missing_files)
    raise FileNotFoundError(
        "Some target matrix files are missing in the external data root. "
        "Inspect the missing files above."
    )

print(
    f"External cell data root is available. "
    f"Checked {len(sanity_rows):,} target cells as a quick sanity check.",
    flush=True,
)
print(
    "Full target/context file availability is checked lazily while loading batches, "
    "which avoids a slow preflight scan of the external volume.",
    flush=True,
)

Checking external cell data root and a small file sample...
External cell data root is available. Checked 20 target cells as a quick sanity check.
Full target/context file availability is checked lazily while loading batches, which avoids a slow preflight scan of the external volume.


In [5]:
print("Checking sampled nonfinite values and computing training normalization statistics...", flush=True)
NONFINITE_SANITY_ROWS = train_df.head(min(20, len(train_df)))
nonfinite_values = summarize_nonfinite_values(NONFINITE_SANITY_ROWS, CELL_DATA_ROOT)
if nonfinite_values.empty:
    print(f"No NaN, -999, or infinite values found in {len(NONFINITE_SANITY_ROWS):,} sampled training target cells.", flush=True)
else:
    display(
        nonfinite_values
        .groupby(["split" if "split" in nonfinite_values.columns else "date", "feature"], dropna=False)
        .agg(files=("path", "count"), nan_values=("nan_count", "sum"), inf_values=("inf_count", "sum"))
    )
    display(nonfinite_values.head(20))

channel_stats_cache = {}
base_channel_stats = compute_channel_stats(
    train_df,
    CELL_DATA_ROOT,
    FEATURE_SUFFIXES,
    context_radius=0,
)


def get_channel_stats(context_radius):
    context_radius = int(context_radius)
    if context_radius not in channel_stats_cache:
        channel_stats_cache[context_radius] = base_channel_stats
    return channel_stats_cache[context_radius]

stats_rows = []
for context_radius in sorted({int(config["context_radius"]) for config in PARAMETER_GRID}):
    mean, std = get_channel_stats(context_radius)
    for feature, feature_mean, feature_std in zip(FEATURE_SUFFIXES, mean, std):
        stats_rows.append(
            {
                "context_radius": context_radius,
                "feature": feature,
                "mean": feature_mean,
                "std": feature_std,
                "estimated_from": "training_target_cells_only",
            }
        )

stats_df = pd.DataFrame(stats_rows)
print("Finished computing normalization statistics.", flush=True)
stats_df

Checking sampled nonfinite values and computing training normalization statistics...
No NaN, -999, or infinite values found in 20 sampled training target cells.
Finished computing normalization statistics.


,context_radius,feature,mean,std,estimated_from
0,0,r,118.646523,47.892353,training_target_cells_only
1,0,g,122.686409,49.060795,training_target_cells_only
2,0,b,126.339005,50.661888,training_target_cells_only
3,0,s,17.467157,14.551807,training_target_cells_only
4,0,d,0.603658,1.108177,training_target_cells_only
5,1,r,118.646523,47.892353,training_target_cells_only
6,1,g,122.686409,49.060795,training_target_cells_only
7,1,b,126.339005,50.661888,training_target_cells_only
8,1,s,17.467157,14.551807,training_target_cells_only
9,1,d,0.603658,1.108177,training_target_cells_only


In [6]:
print("Building dataset/DataLoader factories and reading one sample batch...", flush=True)
def make_datasets(context_radius):
    channel_mean, channel_std = get_channel_stats(context_radius)
    train_dataset = BeachCellDataset(
        train_df,
        CELL_DATA_ROOT,
        class_to_idx,
        image_size=IMAGE_SIZE,
        mean=channel_mean,
        std=channel_std,
        augment=USE_SPATIAL_AUGMENTATION,
        context_radius=context_radius,
    )
    validation_dataset = BeachCellDataset(
        validation_df,
        CELL_DATA_ROOT,
        class_to_idx,
        image_size=IMAGE_SIZE,
        mean=channel_mean,
        std=channel_std,
        context_radius=context_radius,
    )
    test_dataset = BeachCellDataset(
        test_df,
        CELL_DATA_ROOT,
        class_to_idx,
        image_size=IMAGE_SIZE,
        mean=channel_mean,
        std=channel_std,
        context_radius=context_radius,
    )
    return train_dataset, validation_dataset, test_dataset, channel_mean, channel_std


def make_train_validation_loaders(config):
    context_radius = int(config["context_radius"])
    train_dataset, validation_dataset, _, _, _ = make_datasets(context_radius)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    return train_loader, validation_loader


sample_context_radius = int(PARAMETER_GRID[0]["context_radius"])
sample_train_dataset, _, _, sample_mean, sample_std = make_datasets(sample_context_radius)
sample_loader = DataLoader(
    sample_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

x_batch, y_batch = next(iter(sample_loader))
print(f"Sample context radius: {sample_context_radius}", flush=True)
print(f"Input batch shape: {tuple(x_batch.shape)}", flush=True)
print(f"Label batch shape: {tuple(y_batch.shape)}", flush=True)

Building dataset/DataLoader factories and reading one sample batch...
Sample context radius: 0
Input batch shape: (32, 5, 20, 20)
Label batch shape: (32,)


In [7]:
print("Selecting compute device and preparing model factory...", flush=True)
device = get_device()
weights = class_weights(train_df, class_to_idx)

def model_factory(config):
    return build_small_cnn(
        num_classes=len(class_to_idx),
        in_channels=len(FEATURE_SUFFIXES),
        dropout=float(config["dropout"]),
    )

print(f"Device: {device}", flush=True)
print(f"Grid configurations: {len(PARAMETER_GRID)}", flush=True)

Selecting compute device and preparing model factory...
Device: cuda
Grid configurations: 12


In [ ]:
print("Starting grid search training...", flush=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / "small_cnn_best.pt"

grid_result = run_grid_search(
    model_factory=model_factory,
    parameter_grid=PARAMETER_GRID,
    train_loader=None,
    validation_loader=None,
    loader_factory=make_train_validation_loaders,
    class_weights=weights,
    idx_to_class=idx_to_class_name,
    device=device,
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    random_seed=RANDOM_SEED,
)

print("Grid search training finished. Saving best model and result files...", flush=True)
model = grid_result.model
history = grid_result.history
best_config = grid_result.config
validation_report = grid_result.validation_report
best_context_radius = int(best_config["context_radius"])
_, _, test_dataset, channel_mean, channel_std = make_datasets(best_context_radius)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

torch.save(
    {key: value.detach().cpu() for key, value in model.state_dict().items()},
    checkpoint_path,
)

best_config_path = OUTPUT_DIR / "best_config.json"
with best_config_path.open("w", encoding="utf-8") as file:
    json.dump(best_config, file, indent=2)

normalization_stats_path = OUTPUT_DIR / "normalization_stats.csv"
normalization_stats = pd.DataFrame(
    {
        "feature": FEATURE_SUFFIXES,
        "mean": channel_mean,
        "std": channel_std,
        "context_radius": best_context_radius,
    }
)
normalization_stats.to_csv(normalization_stats_path, index=False)

inference_metadata_path = OUTPUT_DIR / "inference_metadata.json"
inference_metadata = {
    "model_type": "small_cnn",
    "image_size": IMAGE_SIZE,
    "context_radius": best_context_radius,
    "context_window_size": 2 * best_context_radius + 1,
    "feature_suffixes": list(FEATURE_SUFFIXES),
    "in_channels": len(FEATURE_SUFFIXES),
    "num_classes": len(class_to_idx),
    "class_to_idx": class_to_idx,
    "idx_to_class_code": idx_to_class_code,
    "class_name_by_code": class_name_by_code,
    "best_config": best_config,
    "channel_mean": [float(value) for value in channel_mean],
    "channel_std": [float(value) for value in channel_std],
}
with inference_metadata_path.open("w", encoding="utf-8") as file:
    json.dump(inference_metadata, file, indent=2)

best_validation_report_path = OUTPUT_DIR / "best_validation_classification_report.csv"
validation_report.to_csv(best_validation_report_path)

grid_search_summary_path = OUTPUT_DIR / "grid_search_summary.csv"
grid_result.summary.to_csv(grid_search_summary_path, index=False)

best_grid_result_path = OUTPUT_DIR / "best_grid_search_result.csv"
grid_result.summary.head(1).to_csv(best_grid_result_path, index=False)

grid_search_metric_trace_path = OUTPUT_DIR / "grid_search_validation_metric_trace.csv"
grid_result.metric_trace.to_csv(grid_search_metric_trace_path, index=False)

history_path = OUTPUT_DIR / "training_history.csv"
history.to_csv(history_path, index=False)

print(f"Best context radius: {best_context_radius}", flush=True)
print("Saved training artifacts. Displaying best grid-search result.", flush=True)
display(grid_result.summary.head(1))
history

Starting grid search training...

Grid search run 1: {'learning_rate': 0.0001, 'weight_decay': 0.0001, 'dropout': 0.25, 'context_radius': 0}


Training:   0%|          | 0/108 [00:00<?, ?it/s]

Validation:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 001 | train loss=1.8158, acc=0.4167 | val loss=1.3784, acc=0.5803


Training:   0%|          | 0/108 [00:00<?, ?it/s]

...

Training:   0%|          | 0/108 [00:00<?, ?it/s]

Validation:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 017 | train loss=0.7173, acc=0.7347 | val loss=0.5879, acc=0.8030


Training:   0%|          | 0/108 [00:00<?, ?it/s]

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [ ]:
print("Saving training history plot...", flush=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_history.png", dpi=150)
plt.show()

In [ ]:
print("Evaluating best model on the test split...", flush=True)
y_true, y_pred, y_probability = predict(model, test_loader, device)
print("Saved test classification outputs.", flush=True)
report_df, confusion_df = classification_metrics(y_true, y_pred, idx_to_class_name)

report_path = OUTPUT_DIR / "test_classification_report.csv"
confusion_path = OUTPUT_DIR / "test_confusion_matrix.csv"

report_df.to_csv(report_path)
confusion_df.to_csv(confusion_path)

report_df

In [ ]:
print("Saving confusion matrix plot...", flush=True)
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(confusion_df.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(confusion_df.columns)))
ax.set_yticks(range(len(confusion_df.index)))
ax.set_xticklabels(confusion_df.columns, rotation=45, ha="right")
ax.set_yticklabels(confusion_df.index)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()

confusion_df

In [ ]:
print("Saving test predictions...", flush=True)
prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_class_code"] = [idx_to_class_code[int(idx)] for idx in y_true]
prediction_df["predicted_class_code"] = [idx_to_class_code[int(idx)] for idx in y_pred]
prediction_df["true_class_name"] = [class_name_by_code[prediction_df.loc[i, "true_class_code"]] for i in range(len(prediction_df))]
prediction_df["predicted_class_name"] = [class_name_by_code[prediction_df.loc[i, "predicted_class_code"]] for i in range(len(prediction_df))]

for idx, class_code in idx_to_class_code.items():
    prediction_df[f"probability_class_{class_code}"] = y_probability[:, idx]

prediction_path = OUTPUT_DIR / "test_predictions.csv"
prediction_df.to_csv(prediction_path, index=False)

print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved best config: {best_config_path}")
print(f"Saved normalization stats: {normalization_stats_path}")
print(f"Saved inference metadata: {inference_metadata_path}")
print(f"Saved best validation report: {best_validation_report_path}")
print(f"Saved grid summary: {grid_search_summary_path}")
print(f"Saved best grid result: {best_grid_result_path}")
print(f"Saved grid metric trace: {grid_search_metric_trace_path}")
print(f"Saved history:    {history_path}")
print(f"Saved report:     {report_path}")
print(f"Saved confusion:  {confusion_path}")
print(f"Saved predictions:{prediction_path}")

prediction_df.head()
print(f"Saved predictions to {prediction_path}", flush=True)